# 配置与导入

In [2]:
import copy
from dataclasses import dataclass

import torch
import torch.nn as nn
import numpy as np
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.models.llama.modeling_llama import LlamaRotaryEmbedding

# 复用和内核一致的 RoPE 应用（与 PaluAttention 实现一致）
from kernel.palu_attention import apply_rotary_pos_emb
from kernel.palu_attention import LlamaPaluAttention

# 超参
MODEL_PATH = "Meta-Llama-3-8B-Instruct_ratio-0.7_gs-4-fisher_uniform-whiten"
SEQ_LEN = 128
BATCH_SIZE = 8
NUM_STEPS = 2000
EVAL_EVERY = 200
MAX_TEST_WINDOWS = 10  # 每次快速 PPL 评估用多少个窗口
DATASET_NAME = "wikitext-2-raw-v1"  # wikitext-2-raw-v1

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# 工具函数：PPL 评估

In [9]:
def evaluate_ppl(model, tokenizer, dataset_name="wikitext2", split="test",
                 seqlen=2048, device="cuda", windows=None):
    import torch
    import torch.nn as nn
    from datasets import load_dataset
    from tqdm import tqdm

    # 与 run_ppl_eval.py 一致的数据来源与切窗方式
    testdata = load_dataset(
        "Salesforce/wikitext",
        "wikitext-2-raw-v1",
        split="test",
    )
    testenc = tokenizer("\n\n".join(testdata["text"]), return_tensors="pt").input_ids

    # 与 run_ppl_eval.py 一致的 forward 与 loss 计算
    model = model.to(device)
    if isinstance(device, str):
        device = torch.device(device)

    nsamples = testenc.numel() // seqlen
    use_cache = model.config.use_cache
    model.config.use_cache = False
    model.eval()

    nlls = []
    with torch.no_grad():
        for i in tqdm(range(nsamples)):
            batch = testenc[:, (i * seqlen):((i + 1) * seqlen)].to(device)
            outputs = model(batch)
            logits = outputs.logits
            shift_logits = logits[:, :-1, :]
            shift_labels = testenc[:, (i * seqlen):((i + 1) * seqlen)][:, 1:].to(device)
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(
                shift_logits.reshape(-1, shift_logits.size(-1)),
                shift_labels.reshape(-1)
            )
            neg_log_likelihood = loss.float() * seqlen
            nlls.append(neg_log_likelihood)

    ppl = torch.exp(torch.stack(nlls).sum() / (len(nlls) * seqlen)).item()
    model.config.use_cache = use_cache
    # example_generation(model, tokenizer, device)
    return ppl


def set_rope_mode(model, *, hack_layer_idx=None):
    # hack_layer_idx 仅在该层打开 latent RoPE，其它层使用 PALU（非 latent）
    for li, layer in enumerate(model.model.layers):
        attn = layer.self_attn
        if isinstance(attn, LlamaPaluAttention):
            attn.rope_latent = (li in hack_layer_idx)

def example_generation(model, tokenizer, device):
    # Example generation (no KV cache to avoid shape mismatch)
    prompt = "Why research is so hard?"
    if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
        tokenizer.pad_token = tokenizer.eos_token

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    model.eval()
    with torch.no_grad():
        prev_use_cache = getattr(model.config, "use_cache", None)
        model.config.use_cache = False  # disable cache globally
        gen_ids = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            use_cache=False,            # disable cache in generate
        )
        if prev_use_cache is not None:
            model.config.use_cache = prev_use_cache

    gen_text = tokenizer.decode(gen_ids[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)

    print("=== Example Prompt ===")
    print(prompt)
    print(gen_text)
    return

# 工具函数 Zero-shot OpenBookQA 准确率评估

In [4]:
def zero_shot_eval(model, tokenizer, tasks, *,
                                      batch_size: int = 8,
                                      max_length: int = 4096,
                                      limit: int | None = None,
                                      return_full: bool = False):
    """
    Same core logic as run_lm_eval.py but uses an already-loaded model/tokenizer.
    - Wraps model/tokenizer with HFLM
    - Runs lm_eval.simple_evaluate on the given tasks
    - Prints the results table and returns results['results'] by default
    """
    import torch
    import lm_eval
    from lm_eval.models.huggingface import HFLM
    from lm_eval.tasks import TaskManager
    from lm_eval.utils import make_table

    # normalize tasks
    task_list = [t.strip() for t in tasks.split(",")] if isinstance(tasks, str) else list(tasks)

    model.seqlen = max_length
    lm_obj = HFLM(pretrained=model, tokenizer=tokenizer, add_bos_token=False, batch_size=batch_size)
    task_manager = TaskManager()

    with torch.no_grad():
        results = lm_eval.simple_evaluate(
            model=lm_obj,
            tasks=task_list,
            task_manager=task_manager,
            log_samples=False,
            limit=limit,
        )

    print(make_table(results))
    return results if return_full else results["results"]
# res = zero_shot_eval(model, tokenizer, tasks=["openbookqa"])

## 1) 加载 模型和dataset（所有层是 PaluAttention，K/V 已分解）

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, torch_dtype=torch.float16, device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("模型已加载。")

# 预缓存测试集以加速 PPL
ds_test = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
test_texts = [ex["text"] for ex in ds_test if ex["text"].strip()]
test_text_cat = "\n\n".join(test_texts)
test_tok = tokenizer(test_text_cat, return_tensors="pt")
test_ids_all = test_tok.input_ids[0]
print("测试集已缓存。")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

模型已加载。
测试集已缓存。


## 2) 评估 Initial Baseline - PPL & OpenBookQA


In [5]:
# 评估 PALU baseline
# print("💪🏻💪🏻💪🏻💪🏻💪🏻💪🏻💪🏻Evaluating LLAMA with Palu-SVD for all layers")
# set_rope_mode(model, hack_layer_idx=[])
# ppl_palu = evaluate_ppl(model, tokenizer, seqlen=2048, windows=10, device=device)
# print(f"PALU(all) PPL: {ppl_palu:.4f}")
# print("\n--------------------------------------------------------------------------------------------------------------------------------\n")

# # 评估仅第id层 HACK
# hack_layer_idx = [0]
# print(f"💪🏻💪🏻💪🏻💪🏻💪🏻💪🏻💪🏻Evaluating LLAMA with HACK-SVD for layer {hack_layer_idx} and Palu-SVD for all other layers")
# set_rope_mode(model, hack_layer_idx=hack_layer_idx)
# ppl_hack0 = evaluate_ppl(model, tokenizer, seqlen=2048, windows=10, device=device)
# print(f"HACK(layer0-only) PPL: {ppl_hack0:.4f}")
# print("\n--------------------------------------------------------------------------------------------------------------------------------\n")
# # 评估完恢复为 PALU
# set_rope_mode(model, hack_layer_idx=[])

#评估zero-shot OpenBookQA准确率
# print("💪🏻💪🏻💪🏻💪🏻💪🏻💪🏻💪🏻Evaluating LLAMA with zero-shot OpenBookQA")
# res = zero_shot_eval(model, tokenizer, tasks=["openbookqa"])

## 3) 存档Palu0, 设置hack0

In [6]:
# 存档palu
if "palu0" not in locals() or palu0 is None:
    palu0  = copy.deepcopy(model.model.layers[0].self_attn)
    assert isinstance(palu0, LlamaPaluAttention), "第0层不是 PaluAttention"
    assert palu0.rope_latent == False, "第0层不是 PALU"
    print("存档 PaluAttention_0")

# 设置hack attention
if "hack0" not in locals() or palu0 is None:
    hack0 = model.model.layers[0].self_attn
    hack0.rope_latent = True  # 目标：RoPE(x@U)@V
    print("配置 HackAttention_0 (rope_latent=True)")

# 只训练 hack0 的 k_proj 中的 U 和 VT
for n, p in hack0.named_parameters():
    p.requires_grad_(False)
train_params = [hack0.k_proj.VT.weight, hack0.k_proj.U[0].weight, hack0.k_proj.U[1].weight]
for p in train_params:
    p.requires_grad_(True)

def reset_model(palu0=None):
    model.model.layers[0].self_attn = palu0.to(device=device)
    hack0 = model.model.layers[0].self_attn.to(device=device)
    hack0.rope_latent = True
    palu0  = copy.deepcopy(model.model.layers[0].self_attn)
    for n, p in hack0.named_parameters():
        p.requires_grad_(False)
    train_params = [hack0.k_proj.VT.weight, hack0.k_proj.U[0].weight, hack0.k_proj.U[1].weight]
    for p in train_params:
        p.requires_grad_(True)

存档 PaluAttention_0
配置 HackAttention_0 (rope_latent=True)


## 5) 构建训练目标：最小化 RoPE(x@U)@V 与 RoPE(x@U@V) 的差异（仅对第0层 K 的 U/V 训练）


In [7]:
optimizer = torch.optim.AdamW(train_params, lr=5e-4, weight_decay=1e-6, eps=1e-8)

# Rotary Embedding（放在第0层所在设备）
attn_dev = hack0.q_proj.weight.device
rotary_full = LlamaRotaryEmbedding(config=hack0.config).to(attn_dev)

# 数据集
ds_train = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")

def sample_batch(tokenizer, batch_size=BATCH_SIZE, seq_len=SEQ_LEN, device=attn_dev):
    texts = []
    while len(texts) < batch_size:
        t = ds_train[np.random.randint(len(ds_train))]["text"].strip()
        if t:
            texts.append(t)
    tok = tokenizer(
        texts, max_length=seq_len, truncation=True, padding="max_length", return_tensors="pt"
    )
    return tok.input_ids.to(device)

@torch.no_grad()
def get_hidden_normed(input_ids):
    # 仅第0层的前处理：embed -> input_layernorm
    embed = model.model.embed_tokens
    ln0 = model.model.layers[0].input_layernorm
    hs = embed(input_ids)
    hs = ln0(hs)
    return hs  # [B, T, H]

def alignment_loss(input_ids):
    B, T = input_ids.shape
    pos_ids = torch.arange(T, device=attn_dev).unsqueeze(0).expand(B, -1)

    # 预处理 hidden_states
    hs = get_hidden_normed(input_ids)  # [B, T, H]

    # 计算 cos/sin（按 K 的 head_dim）
    # 注意：rotary_full 的前向需要一个“形状提示”，这里用 K 的最终形状来生成 cos/sin
    # 用一个 dummy tensor 只为生成 cos/sin；下方实际应用在不同张量上
    head_dim = hack0.head_dim
    num_kv = hack0.num_key_value_heads
    dummy = torch.empty(B, num_kv, T, head_dim, device=attn_dev, dtype=hs.dtype)
    cos, sin = rotary_full(dummy, pos_ids)  # 形状 [B, T, head_dim]

    # === 目标：PALU 路径（RoPE(x@U@V)) ===
    with torch.no_grad():
        k_lat_palu = palu0.k_proj.project_to_latent(hs)  # [B, T, total_latent_k]
        k_palu = palu0.k_proj.reconstruct(k_lat_palu)    # [B, T, num_kv*head_dim]
        k_palu = k_palu.view(B, T, num_kv, head_dim).transpose(1, 2)  # [B, heads, T, head_dim]
        _, k_palu_rope = apply_rotary_pos_emb(None, k_palu, cos, sin)  # 对 key 施加 RoPE

    # === 预测：HACK 路径（RoPE(x@U)@V) ===
    k_lat_hack = hack0.k_proj.project_to_latent(hs.float())  # [B, T, total_latent_k]
    latent_dim = k_lat_hack.shape[-1] // num_kv
    k_lat_hack = k_lat_hack.view(B, T, num_kv, latent_dim).transpose(1, 2)  # [B, heads, T, latent_dim]
    # 在 latent 维度上截断 cos/sin 后应用 RoPE
    _, k_lat_hack_rope = apply_rotary_pos_emb(None, k_lat_hack, cos[..., :latent_dim], sin[..., :latent_dim])
    # 重构回 key states
    k_lat_hack_rope = k_lat_hack_rope.transpose(1, 2).reshape(B, T, -1)  # [B, T, total_latent_k]
    k_hack = hack0.k_proj.reconstruct(k_lat_hack_rope).view(B, T, num_kv, head_dim).transpose(1, 2)  # [B, heads, T, head_dim]

    # MSE 对齐
    return nn.functional.mse_loss(k_hack, k_palu_rope.float())


## 6) 训练循环：优化 U/V（仅 K），并周期性评估整模 PPL

In [8]:
def quick_ppl(model, tokenizer, seqlen=2048, windows=MAX_TEST_WINDOWS):
    model.eval()
    prev_use_cache = getattr(model.config, "use_cache", None)
    model.config.use_cache = False

    vocab = model.lm_head.out_features
    nlls = []
    used = 0
    max_end = test_ids_all.shape[0] - seqlen - 1
    if max_end <= 0:
        if prev_use_cache is not None:
            model.config.use_cache = prev_use_cache
        # print("[quick_ppl] not enough tokens for seqlen.")
        return float("nan")

    with torch.no_grad():
        for i in range(0, max_end, seqlen):
            batch = test_ids_all[i:i+seqlen].unsqueeze(0).to(device)
            target = test_ids_all[i+1:i+seqlen+1].unsqueeze(0).to(device)
            try:
                out = model(batch, use_cache=False)
                logits = out.logits
                # 过滤 NaN/Inf logits
                if torch.isnan(logits).any() or torch.isinf(logits).any():
                    print("logits is nan or inf")
                    print(logits)
                    continue
                # 在 fp32 上算交叉熵更稳
                loss = nn.functional.cross_entropy(
                    logits.float().view(-1, logits.size(-1)),
                    target.view(-1),
                    reduction="mean"
                )
            except Exception:
                continue

            if torch.isfinite(loss):
                nlls.append(loss.item())
                used += 1
                if used >= windows:
                    break

    if prev_use_cache is not None:
        model.config.use_cache = prev_use_cache

    if not nlls:
        print(f"[quick_ppl] no valid windows (used={used}, vocab={vocab}).")
        return float("nan")
    return float(np.exp(np.mean(nlls)))

# 初始 PPL
hack0.to(dtype=torch.float16)
base_ppl = evaluate_ppl(model, tokenizer, DATASET_NAME, "test", 2048, device)
hack0.to(dtype=torch.float32)
print(f"Baseline (HACK@layer0) PPL: {base_ppl:.4f}")

loss_hist = []
ppl_hist = []
best_ppl = float('inf')
best_layer0 = None
for step in tqdm(range(1, NUM_STEPS + 1), desc="Aligning K (RoPE latent vs full)"):
    input_ids = sample_batch(tokenizer, BATCH_SIZE, SEQ_LEN, attn_dev)
    
    optimizer.zero_grad(set_to_none=True)
    hack0.to(dtype=torch.float32)
    loss = alignment_loss(input_ids)
    if torch.isfinite(loss):
        loss.backward()
        torch.nn.utils.clip_grad_norm_(train_params, 0.05)
        optimizer.step()
        loss_hist.append(float(loss.item()))
    hack0.to(dtype=torch.float16)
    if step % EVAL_EVERY == 0 or step <= 10:
        ppl = quick_ppl(model, tokenizer, seqlen=2048, windows=MAX_TEST_WINDOWS)
        ppl_hist.append(ppl)
        print(f"Step {step}: align_loss={loss.item():.6e}, quick PPL={ppl:.4f}")
        if ppl < best_ppl:
            best_ppl = ppl
            best_layer0 = copy.deepcopy(model.model.layers[0].self_attn)

# 结束后做一次完整 PPL
model.model.layers[0].self_attn = best_layer0
final_ppl = evaluate_ppl(model, tokenizer, DATASET_NAME, "test", 2048, device)
print(f"Final (HACK@layer0) PPL: {final_ppl:.4f}")

  0%|                                                                                             | 0/10 [00:00<?, ?it/s]

😄😄😄😄😄😄😄😄😄😄😄No KV Cache


100%|████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:02<00:00,  4.29it/s]


Baseline (HACK@layer0) PPL: 1973.3740


Aligning K (RoPE latent vs full):   0%|                                               | 1/2000 [00:02<1:11:04,  2.13s/it]

...

Step 1800: align_loss=1.926775e-01, quick PPL=839.5547


Aligning K (RoPE latent vs full): 100%|██████████████████████████████████████████████| 2000/2000 [00:48<00:00, 41.11it/s]

Step 2000: align_loss=1.436938e-01, quick PPL=855.4701



100%|████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:01<00:00,  5.12it/s]


Final (HACK@layer0) PPL: 17.0997


In [10]:
print(evaluate_ppl(model, tokenizer, DATASET_NAME, "test", 2048, device))

100%|██████████████████████████████████████████████████████████████████████████████████| 141/141 [00:27<00:00,  5.09it/s]


19.658885955810547
